## Part C1.2. Computing Nematic order using TrackMate-extracted spot data
#### The objective of this jupyter notebook is to extract the relative orientation of single cells to compute the nematic order for each condition. The data processing was based on the segmented masks retrieved using Onimpose in Part A. 

#### Subsequently, we applied TrackMate using the same methods described in Part B to analyze the information of cells (spots) in the colony. It is noted that the properties of the mask was set with 0.065 um pixel size and 5-second frame interval. The TrackMate-extracted CSV file for each condition was save and directly perform batch processing, demonstrated here in this code.

In [91]:
#Import essential libraries 
import pandas as pd
import numpy as np
import os

"""
Perform batch processing for multiple CSV files corresponding to different mutants and densities.
This code will finally save the results in a single CSV file that includes 
all the Nematic order values for all 9 conditions 
"""
# Define the directory where the CSV files are located. It's noted that this depends on the data storage of the user
data_directory = '/Users/herbert/Desktop/ME480/Projects/Project1/Data/Trackmate_output'

# Obtain a list of all CSV files in the directory using os.listdir()
file_names = [f for f in os.listdir(data_directory) if f.endswith ('csv')]

### Define the function to calculate the Nematic Order
##### Equation1: Sr_2 = =<cos[2(θ_i−ϕ_i)]>

In [82]:
#Define a function to calculate the Nematic order
def calculate_nematic_order(theta, phi):
    """
    Calculates the nematic order parameter S_R for a group of cells (i).
    
    :param theta: Array of angular orientations (theta_i) in radians
    :param phi: Array of angular positions (phi_i) in radians (polar coordinates about colony center)
    :return: Nematic order parameter (S_R)
    """
    diff = theta - phi  # Difference between theta_i and phi_i
    nematic_order = np.mean(np.cos(2 * diff))
    return nematic_order

# Function to calculate polar angle (phi) from X and Y coordinates
def calculate_phi(x, y, center_x, center_y):
    """
    Calculates the polar angle phi_i for each bacterium relative to the colony center.
    
    :param x: X-coordinate of bacterium
    :param y: Y-coordinate of bacterium
    :param center_x: X-coordinate of the colony center
    :param center_y: Y-coordinate of the colony center
    :return: Array of phi_i values (in radians)
    """
    return np.arctan2(y - center_y, x - center_x)  # Calculate phi_i using arctan2 for full 360° coverage



### C1.2.1. Calculate the Nematic Order of the Whole Region for All Frame: 
##### To calculate the nematic order for a given list of Ellipse Theta(in radians). 

In [83]:
# Initialize an empty list to hold the results
all_results = []

# Group the data by time and calculate nematic order for each time frame
nematic_order_per_timeframe = []

# Loop over each CSV file, process it and calcualate its Nematic Order
for file_name in file_names:
    file_path = os.path.join(data_directory, file_name)
    #reads the entire file but aviod guessing data types based on partial chunks
    data = pd.read_csv(file_path, low_memory = False)

    #To handle the Trackmate-extracted data
    #Ensure all columns are numeric, forcing non-numeric to NaN
    data['POSITION_X'] = pd.to_numeric(data['POSITION_X'], errors='coerce')
    data['POSITION_Y'] = pd.to_numeric(data['POSITION_Y'], errors='coerce')
    data ['POSITION_T'] = pd.to_numeric (data['POSITION_T'], errors = 'coerce')
    data ['ELLIPSE_THETA'] = pd.to_numeric (data['ELLIPSE_THETA'], errors = 'coerce')

    # Drop rows with NaN values (due to non-numeric entries)
    data = data.dropna(subset=['POSITION_X','POSITION_Y','POSITION_T','ELLIPSE_THETA'])

    # Inspect the first few rows of the data (optional)
    #print(data.head())
    
    # Process each Time frame in the CSV file 
    # Group data by Time (sec) and apply the calculation for each group
    for timeframe, group in data.groupby('POSITION_T'): # The name for the columne of Time 
        x_coords = group ['POSITION_X'].values # X coordinates in um
        y_coords = group ['POSITION_Y'].values # Y coordinates in um
        
        center_x = np.mean (x_coords)
        center_y = np.mean (y_coords)
        # print the colony center (optional)
        # print ('colony center (x,y):' , center_x, ",", center_y) 
        
        #Extract the 'Ellipse angle' values for each time frame 
        theta_values = group ['ELLIPSE_THETA'].values # The name for the columne of Ellipse angle

        phi_values = calculate_phi(x_coords, y_coords, center_x, center_y)
        
        nematic_order = calculate_nematic_order (theta_values, phi_values)

        # Append the reulst with condition name, Time , and Nematic order
        condition_name = file_name.split('.')[0] #Extract the condition name from the file name
        # Store the result as a tuple (position_T (time), nematic order)    
        all_results.append((condition_name, timeframe, nematic_order))
    
# Convert the results to a pandas DataFrame for saving
final_df = pd.DataFrame (all_results, columns=['Condition', 'Timeframe', 'Nematic Order'])

# Demonstrate the results from calculation (optional)
# print("Sr_calculated as", all_results)

# Save the results to a new CSV file
output_file = '4_nematic_order_all.csv' #The file name should be changed based on the experimental dataset 
final_df.to_csv(output_file, index=False)

print("Nematic order for all conditions calculated and saved to", output_file)

Nematic order for all conditions calculated and saved to 1_nematic_order_all.csv


### C.1.2.2. Calculate the Nematic Order of the Region of Interest using *Distance-based Filter*

#### From the previous calculation (C1.2.1.), we noticed that the calculated Numatic order is independent to frame. Ideally, we could reduce the frame number to save computational power (optional). 

#### More importantly, from what we have learned in the leacture, calculating Numatic Order should be defined to a certain region that only considers such order parameter among the neighbouring bacteria. The next two methods aim to initiatelly define a certain region in a relatively close proximity before calculating the Numatic order, so as to improve the accuratcy of the caulcation.

#### In this part, we consider only neighboring cells within a certain distance by incorporating a distance-base filtering mechanmism. Specifically, here only the cells within the specificed 30 um will be considered when calculating the Mean Nematic order using the X and Y coordinates provided from the TrackMate results.

In [92]:
from scipy.spatial import distance

# The distance threshold was preliminary determined by drawing lines in FIJI to measure the averaged distance between cells
max_distance = 30  # 30 µm distance for considering neighboring cells

# define a fuction to compute pairwise distances and filter neighbors 

def filter_neighbors_by_distance(x, y, i, max_distance):
    """
    Filters neighboring cells based on their distance from the cell at index i.
    
    :param x: Array of X-coordinates of cells
    :param y: Array of Y-coordinates of cells
    :param i: Index of the current cell
    :param max_distance: Maximum distance to consider as a neighbor
    :return: Boolean mask indicating which cells are within max_distance of cell i
    """
    distances = np.sqrt((x - x[i])**2 + (y - y[i])**2)
    neighbors_mask = (distances <= max_distance) & (distances > 0)  # Exclude the cell itself (distances > 0)
    return neighbors_mask

def calculate_nematic_order_for_neighbors2(theta, x, y, max_distance):
    """
    Calculate nematic order parameter for each cell, considering only neighboring cells within max_distance.
    
    :param theta: Array of angular orientations (theta_i)
    :param phi: Array of angular positions (phi_i)
    :param x: Array of X-coordinates of cells
    :param y: Array of Y-coordinates of cells
    :param max_distance: Maximum distance for considering neighboring cells
    :return: The mean of nematic order parameters for each frame
    """
    num_cells = len(x)  # Total number of cells
    processed_cells = set()  # Track already processed cells to avoid duplication
    nematic_order_sum = 0  # Store nemaitc order for each frame
    count_neighbors = 0

    # Loop through each cell
    for i in range(num_cells):
        if i in processed_cells:
            continue  # Skip cells that have already been processed

        # Find neighbors within max_distance of cell i
        neighbors_mask = filter_neighbors_by_distance(x, y, i, max_distance)
        
        # Extract the orientations and phi of neighbors
        neighbor_orientations = theta[neighbors_mask]
        neighbor_x = x[neighbors_mask]
        neighbor_y = y[neighbors_mask]

        # If no neighbors are found, skip this cell
        if len(neighbor_orientations) == 0:
            continue
            
        # Dynamically calculate the centriod (new center) of the neighboring cells
        center_x = np.mean(neighbor_x)
        center_y = np.mean(neighbor_y)

        #Recalculate phi values for the neighbouring cells relative to the new center
        neighbor_phis = calculate_phi (neighbor_x, neighbor_y, center_x, center_y)

        # Calculate nematic order for this cell's neighbors
        nematic_order = np.mean(np.cos(2 * (neighbor_orientations - neighbor_phis)))
        nematic_order_sum += nematic_order * len(neighbor_orientations)
        count_neighbors += len(neighbor_orientations)

        # Mark the current cell and its neighbors as processed
        processed_cells.update(np.where(neighbors_mask)[0])


    # Compute the mean radial order across all neighbors (avoid division by zero)
    if count_neighbors > 0:
        mean_nematic_order = nematic_order_sum/count_neighbors
    else:
        mean_nematic_order = np.nan  # Handle case with no neighbors
    
    return mean_nematic_order

# Initialize an empty list to hold the results
all_results = []

# Loop over each CSV file, process it and calcualate its Nematic Order
for file_name in file_names:
    file_path = os.path.join(data_directory, file_name)
    
    #reads the entire file but aviod guessing data types based on partial chunks
    data = pd.read_csv(file_path, low_memory = False)

    # Ensure X, Y, T, and Theta columns are numeric, forcing non-numeric to NaN
    data['POSITION_X'] = pd.to_numeric(data['POSITION_X'], errors='coerce')
    data['POSITION_Y'] = pd.to_numeric(data['POSITION_Y'], errors='coerce')
    data['POSITION_T'] = pd.to_numeric(data['POSITION_T'], errors='coerce')
    data['ELLIPSE_THETA'] = pd.to_numeric(data['ELLIPSE_THETA'], errors='coerce')

    # Drop rows with NaN values (due to non-numeric entries)
    data = data.dropna(subset=['POSITION_X','POSITION_Y','POSITION_T','ELLIPSE_THETA'])
    
    # Process each Time frame in the CSV file 
    # Group data by Time (sec) and apply the calculation for each group
    for timeframe, group in data.groupby('POSITION_T'): #The units for the columne of Time 

        x_values = group ['POSITION_X'].values
        y_values = group ['POSITION_Y'].values
        theta_values = group['ELLIPSE_THETA'].values
        
        # Using the 
        mean_nematic_order = calculate_nematic_order_for_neighbors2 (theta_values, x_values, y_values, max_distance)
        
        # Append the result with condition name (extracted from file_name), frame, and Nematic order
        condition_name = file_name.split('.')[0]  # Extract condition name from the file name (before '.csv')
        all_results.append((condition_name, timeframe, mean_nematic_order))

# Convert the accumulated results to a pandas DataFrame
final_df = pd.DataFrame(all_results, columns=['Condition', 'Timeframe', 'Nematic Order'])

# Save the all results to a single CSV file
output_file = '2_Sr_by_distance.csv'
final_df.to_csv(output_file, index=False)

print(f"Nematic order for all conditions (within 30 µm) saved to {output_file}")

Nematic order for all conditions (within 30 µm) saved to 2_Sr_by_distance.csv


### C.1.2.3. Calculate the Nematic Order of the Region of Interest using Region Filter

#### This part aims to first define the region of interest (ROI).We divided the field of view into ROI and compuate the nematic order within each region by dividing the 133x133 um filed into 4 quadrants. For each region, the data was filtered based on X and Y coordinates to restrict the nematic order calculation to cells within these regions. 

In [79]:
fov_size = 133.0  # Field of view size in µm (133x133 µm)
half_fov = fov_size / 2  # 66.5 µm, used for quadrant splitting

# Step 4: Define function to filter cells by region
def filter_by_region(data, region):
    """
    Filters cells based on their X, Y coordinates to fit within a given region (quadrant).
    
    :param data: Pandas DataFrame containing X, Y, and Theta columns
    :param region: String indicating the region ('Q1', 'Q2', 'Q3', 'Q4')
    :return: Filtered DataFrame containing only cells within the region
    """
    if region == 'Q1':  # Top-left
        return data[(data['POSITION_X'] < half_fov) & (data['POSITION_Y'] >= half_fov)]
    elif region == 'Q2':  # Top-right
        return data[(data['POSITION_X'] >= half_fov) & (data['POSITION_Y'] >= half_fov)]
    elif region == 'Q3':  # Bottom-left
        return data[(data['POSITION_X'] < half_fov) & (data['POSITION_Y'] < half_fov)]
    elif region == 'Q4':  # Bottom-right
        return data[(data['POSITION_X'] >= half_fov) & (data['POSITION_Y'] < half_fov)]
    else:
        return pd.DataFrame()  # Return empty DataFrame if region is invalid

# Step 5: Initialize an empty list to hold the results
all_results = []

# Step 6: Loop over each CSV file, process it, and calculate Nematic order for each quadrant
for file_name in file_names:
    file_path = os.path.join(data_directory, file_name)  # Get full file path
    data = pd.read_csv(file_path, low_memory=False)  # Read CSV with low_memory=False

    # Ensure all columns are numeric, forcing non-numeric to NaN
    data['POSITION_X'] = pd.to_numeric(data['POSITION_X'], errors='coerce')
    data['POSITION_Y'] = pd.to_numeric(data['POSITION_Y'], errors='coerce')
    data['POSITION_T'] = pd.to_numeric(data['POSITION_T'], errors='coerce')
    data['ELLIPSE_THETA'] = pd.to_numeric(data['ELLIPSE_THETA'], errors='coerce')

    # Drop rows with NaN values (due to non-numeric entries)
    data = data.dropna(subset=['POSITION_X', 'POSITION_Y', 'POSITION_T','ELLIPSE_THETA'])

    # Process each frame in the CSV file
    for timeframe, group in data.groupby('POSITION_T'):

        x_values = group ['POSITION_X'].values
        y_values = group ['POSITION_Y'].values
        theta_values = group['ELLIPSE_THETA'].values
        
        # Loop through each quadrant and calculate nematic order for cells in that quadrant
        for quadrant in ['Q1', 'Q2', 'Q3', 'Q4']:
             # Define the colony center based on the quadrant, given that (0,0) is at bottom left corner for TrackMate-based data
            if quadrant == 'Q1':
                center_x = half_fov / 2  # Center for Q1 (top-left)
                center_y = (3 * half_fov) / 2
            elif quadrant == 'Q2':
                center_x = (3 * half_fov) / 2  # Center for Q2 (top-right)
                center_y = (3 * half_fov) / 2
            elif quadrant == 'Q3':
                center_x = half_fov / 2  # Center for Q3 (bottom-left)
                center_y = half_fov / 2
            elif quadrant == 'Q4':
                center_x = (3 * half_fov) / 2  # Center for Q4 (bottom-right)
                center_y = half_fov / 2
            filtered_data = filter_by_region(group, quadrant)

            # Only calculate Nematic order if there are cells in the quadrant
            if len(filtered_data) > 0:
                theta_values = filtered_data['ELLIPSE_THETA'].values  # Orientation (Theta)

                #Calculate phi for each cell relative to colony center
                phi_values = calculate_phi(filtered_data['POSITION_X'].values, filtered_data['POSITION_Y'].values, center_x, center_y)
        
                nematic_order = calculate_nematic_order(theta_values, phi_values)
            else:
                nematic_order = np.nan  # If no cells, set as NaN
            
            # Append the result with condition name, frame, quadrant, and Nematic order
            condition_name = file_name.split('.')[0]  # Extract condition name from the file name (before '.csv')
            all_results.append((condition_name, timeframe, quadrant, nematic_order))

# Step 7: Convert the accumulated results to a pandas DataFrame
final_df = pd.DataFrame(all_results, columns=['Condition', 'TimeFrame', 'Quadrant', 'Nematic Order'])

# Step 8: Save the combined results to a single CSV file
output_file = '3_Sr_by_quadrant.csv'
final_df.to_csv(output_file, index=False)

print(f"Nematic order for all conditions by quadrant saved to {output_file}")

Nematic order for all conditions by quadrant saved to 6_Sr_by_quadrant.csv


### C.1.2.4